# Event Detection

Rule-based event extraction (passes, carries, shots, possession) from the
persisted **game state** produced by `src.pipeline`.

**Status:** Working end-to-end. Validated on `sut-mla` (60s, then a 4-min window)
with visual QC passes (renders in `output/qc/sut-mla/`).

## Pipeline
```
broadcast video
  → src.pipeline.PerceptionPipeline   (YOLO + BoT-SORT + team + PnLCalib homography)
      → output/game_state/{slug}/      (players/frames/ball parquet + meta.json)
  → src.ball_tracker.track_ball        (pitch-space Kalman + hindsight bridging)
  → src.events.detect_events           (possession → touches → events)
      → output/events/{slug}_events.json    (StatsBomb-lite)
```

The game state is the **cache-once keystone**: perception (slow, GPU) runs once
and is persisted; ball tracking + event iteration below are instant and need no video.

## Method
Possession-then-event decision tree (Anzer/Bauer, PLOS One 2024): decide the ball
carrier each frame → collapse into discrete *touches* → classify each touch→touch
transition. Events are emitted **only on trusted frames**
(`is_wide_shot AND homog_conf ≥ HOMOG_CONF_MIN`) — visual QC showed the raw `has_P`
coverage was inflated by wrong projections on close-ups and a mid-confidence band.

## Ball state (Milestone 2 — done)
YOLO loses the ball on ~45% of frames (12–64-frame gaps, exactly during passes).
`src.ball_tracker` replaces the old damped extrapolation with a **pitch-space
constant-velocity Kalman filter** over the persisted candidates (`ball.parquet`):
Mahalanobis-gated, motion-consistent detection selection, birth/confirm hysteresis,
kick reinit, honest coasting (a ball unseen > `CARRIER_MAX_MISSED` frames cannot
extend possession), plus a **hindsight bridging pass** that rewrites coast runs
bounded by detections ≤ 2.4 s apart as a straight line (`source='bridged'`) —
visually verified to land within ~1–2 m of the real ball mid-blackout.

In [ ]:
import sys
import json
import importlib
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import src.game_state, src.ball_tracker, src.events, src.pipeline
for _m in (src.game_state, src.ball_tracker, src.events, src.pipeline):
    importlib.reload(_m)

from src.config import Config
from src.game_state import GameState
from src.ball_tracker import track_ball, coverage_report, BallTrackerParams
from src.events import (detect_events, export_events,
                        ball_series, carrier_per_frame, build_touches)
from src.pipeline import PerceptionPipeline

print('Loaded.')

In [2]:
GAME_SLUG = 'sut-mla'   # <- change per game
PERIOD = 1              # <- per-half artifacts: 1 or 2

# The game state is built once from the CLI (recommended — it's a long GPU pass):
#     python -m src.pipeline --match sut-mla --offset_min 10 --duration_sec 60
#
# ...or build it inline (uncomment). period_info gives the frame ranges.
# pipe = PerceptionPipeline(GAME_SLUG)
# p, fps = pipe.period_info, pipe.fps
# start = p['first_half_start_frame'] + int(10 * 60 * fps)
# gs = pipe.run(start, start + int(60 * fps), period=1)

gs = GameState.load(GAME_SLUG, period=PERIOD)
print(f"{GAME_SLUG}: {len(gs.frames)} frames, {len(gs.players)} player-rows, fps {gs.fps}")
print(f"period {gs.meta.get('period')} | homography sources: {gs.meta.get('homog_source_counts')}")
with_p = int(gs.frames['has_P'].sum())
print(f"frames with valid projection: {with_p} ({100*with_p/len(gs.frames):.1f}%)")

sut-mla: 1500 frames, 20125 player-rows, fps 25.0
period 1 | homography sources: {'none': 51, 'manual_seed': 34, 'stale': 119, 'pnlcalib': 1270, 'flow': 26}
frames with valid projection: 1449 (96.6%)


## Step 0 — QC: ball track

Run the Kalman tracker standalone and eyeball the trajectory before trusting the
events built on it. Green = detected, blue = hindsight-bridged, orange = causal
coast. Strands should be continuous (no teleports); bridges should connect
detection strands in straight pass-like lines.

In [ ]:
ball = track_ball(gs)
print(coverage_report(gs, ball))

fig, ax = plt.subplots(figsize=(12, 7.5))
ax.set_xlim(-3, 108); ax.set_ylim(71, -3); ax.set_aspect('equal'); ax.axis('off')
ax.add_patch(plt.Rectangle((0, 0), 105, 68, fill=False, ec='gray'))
ax.plot([52.5, 52.5], [0, 68], color='gray', lw=1)
ax.add_patch(plt.Circle((52.5, 34), 9.15, fill=False, ec='gray'))
for x0 in (0, 88.5):
    ax.add_patch(plt.Rectangle((x0, 13.84), 16.5, 40.32, fill=False, ec='gray'))
for src_name, col in (('detected', 'green'), ('bridged', 'dodgerblue'),
                      ('kalman', 'orange')):
    sub = ball[ball.source == src_name]
    ax.scatter(sub.x, sub.y, s=4, c=col, label=f'{src_name} ({len(sub)})')
ax.legend(loc='upper left')
ax.set_title(f'{GAME_SLUG} — Kalman ball track by source (pitch metres)')
plt.show()

# Speed profile: kicks show as spikes; sustained >30 m/s would be suspicious.
fig, ax = plt.subplots(figsize=(12, 2.2))
ax.plot(ball.time_sec, ball.speed, lw=0.7)
ax.set_ylabel('ball speed (m/s)'); ax.set_xlabel('match time (s)')
plt.show()

## Step 1 — Detect events

Runs the whole possession → touches → events chain on the cached game state.
Instant — no video, no GPU.

In [3]:
events, summary = detect_events(gs)
print(json.dumps(summary, indent=2))
print(f"\n{len(events)} events:")
for e in events[:15]:
    extra = {k: e.details[k] for k in ('recipient', 'outcome', 'length', 'from_team')
             if k in e.details}
    print(f"  {e.time_sec:6.1f}s  {e.type:18s} team{e.team} p{e.player:<4} {extra}")

{
  "n_events": 22,
  "homog_conf_min": 0.75,
  "homography_trusted_pct": 77.5,
  "carrier_frames": 757,
  "possession_pct": {
    "0": 98.0,
    "1": 2.0
  },
  "passes": {
    "0": 10,
    "1": 1
  },
  "pass_completion_pct": {
    "0": 100.0,
    "1": 0.0
  },
  "carries": {
    "0": 7,
    "1": 0
  },
  "shots": {
    "0": 0,
    "1": 0
  },
  "possession_changes": 4
}

22 events:
   606.3s  Carry              team0 p89   {'length': 5.2}
   609.0s  Pass               team0 p89   {'recipient': 201, 'outcome': 'complete', 'length': 27.92}
   612.2s  Possession Change  team1 p207  {'from_team': 0}
   612.7s  Possession Change  team0 p201  {'from_team': 1}
   612.8s  Possession Change  team1 p151  {'from_team': 0}
   612.9s  Pass               team1 p151  {'outcome': 'interception', 'length': 8.67}
   613.8s  Possession Change  team0 p86   {'from_team': 1}
   614.7s  Pass               team0 p86   {'recipient': 304, 'outcome': 'complete', 'length': 15.7}
   616.5s  Pass               t

## Step 2 — QC: pass map & possession

Visual sanity check on the pitch (metres). Passes should read like real build-up
play — no wild diagonals to nowhere, both teams represented in contested spells.
Solid arrow = completed pass, dashed = lost/intercepted, ★ = shot.

In [4]:
def draw_pitch(ax):
    ax.set_xlim(-3, 108); ax.set_ylim(-3, 71); ax.set_aspect('equal'); ax.axis('off')
    ax.add_patch(plt.Rectangle((0, 0), 105, 68, fill=False, ec='gray'))
    ax.plot([52.5, 52.5], [0, 68], color='gray', lw=1)
    ax.add_patch(plt.Circle((52.5, 34), 9.15, fill=False, ec='gray'))
    for x0 in (0, 88.5):
        ax.add_patch(plt.Rectangle((x0, 13.84), 16.5, 40.32, fill=False, ec='gray'))

# team colours matched to kits (QC-verified): team0 = Mladost (yellow), team1 = Sutjeska (blue)
TC = {0: 'goldenrod', 1: 'tab:blue', 2: 'gray'}
NAME = {0: 'Mladost', 1: 'Sutjeska'}

fig, ax = plt.subplots(figsize=(12, 7.5)); draw_pitch(ax)
for e in events:
    if e.type == 'Pass' and 'end_location' in e.details:
        (x, y), (ex, ey) = e.location, e.details['end_location']
        done = e.details.get('outcome') == 'complete'
        ax.annotate('', xy=(ex, ey), xytext=(x, y),
                    arrowprops=dict(arrowstyle='->', color=TC[e.team], lw=2,
                                    alpha=0.9 if done else 0.45,
                                    linestyle='-' if done else '--'))
    elif e.type == 'Carry' and 'end_location' in e.details:
        (x, y), (ex, ey) = e.location, e.details['end_location']
        ax.plot([x, ex], [y, ey], color=TC[e.team], lw=1, ls=':', alpha=0.6)
    elif e.type == 'Shot':
        ax.scatter([e.location[0]], [e.location[1]], c='red', marker='*', s=260, zorder=6)
ax.set_title(f"{GAME_SLUG} — passes (solid=complete, dashed=lost), carries (dotted), shots (star)")
plt.show()

# Possession bar
p = summary['possession_pct']
fig, ax2 = plt.subplots(figsize=(8, 1.1))
ax2.barh([0], [p[0]], color=TC[0]); ax2.barh([0], [p[1]], left=[p[0]], color=TC[1])
ax2.set_xlim(0, 100); ax2.set_yticks([])
ax2.set_title(f"Possession   {NAME[0]} {p[0]}%   |   {NAME[1]} {p[1]}%   "
              f"(passes {NAME[0]} {summary['passes'][0]} / {NAME[1]} {summary['passes'][1]})  "
              f"| trusted homography {summary['homography_trusted_pct']}%")
plt.show()

C:\Users\PC\AppData\Local\Temp\ipykernel_7896\3351716089.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\PC\AppData\Local\Temp\ipykernel_7896\3351716089.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 3 — Export (StatsBomb-lite JSON)

Writes `output/events/{slug}_events.json`: a `summary` block plus per-event
records with `location` in StatsBomb 120×80 coords, native `pitch_xy` in metres,
and a type-specific detail block (`pass`/`carry`/`shot`/`possession_change`).

In [5]:
path = export_events(events, GAME_SLUG, summary)
print('Saved', path)

# Peek at the first few exported records
exported = json.loads(Path(path).read_text())
for rec in exported['events'][:5]:
    print(rec['timestamp'], rec['type'], 'team', rec['team'],
          'loc', rec['location'])

Saved C:\Users\PC\Desktop\GitHub\football-computer-vision\output\events\sut-mla_events.json
10:06.320 Carry team 0 loc [23.4, 10.85]
10:08.960 Pass team 0 loc [31.8, 6.89]
10:12.240 Possession Change team 1 loc [61.33, 18.19]
10:12.720 Possession Change team 0 loc [58.87, 17.89]
10:12.840 Possession Change team 1 loc [56.52, 19.48]
